# MLflow 3.13 GenAI Workshop

**Hands-on lab:** Prompt Registry · AI Gateway · Pydantic AI · Evaluation · Optimization

We build a small **support ticket triage** app end-to-end:

1. Store and version prompts in the **Prompt Registry**
2. Call models through the **AI Gateway** (no provider keys in app code)
3. Wire a **Pydantic AI** agent to the gateway + registry
4. **Evaluate** prompt quality on a labeled dataset
5. **Optimize** the prompt and promote the winner with aliases

> **Workshop tip:** Run cells top-to-bottom. After Part 1, open the MLflow UI → **Prompts** tab to inspect what you registered.

## 0. Prerequisites

Before starting, ensure you have:

1. **Python 3.12+** and project dependencies installed:
   ```bash
   uv sync
   ```
2. **Environment file** — copy and edit `Docker.env` in the project root:
   ```bash
   cp Docker.env Docker.env.local   # optional backup
   # edit MLFLOW_TRACKING_URI, MLFLOW_GATEWAY_ENDPOINT, MLFLOW_WORKSPACE, ...
   ```
   The notebook loads `Docker.env` automatically via `python-dotenv`.
3. **MLflow server + AI Gateway** running. Either:
   - Start from the example config:
     ```bash
     cp gateway_config.example.yaml gateway_config.yaml
     mlflow gateway start --config-path gateway_config.yaml --host 127.0.0.1 --port 5000
     ```
   - Or use an existing server (set `MLFLOW_GATEWAY_ENDPOINT` in `Docker.env`).

4. **Workspaces (optional today, ready for later):** set `MLFLOW_WORKSPACE=default` in `Docker.env`.
   When your server runs with `--enable-workspaces`, all registry / experiment / eval operations
   are scoped to that workspace. On servers without workspaces, the notebook detects this and
   runs in legacy mode automatically.

5. Open the MLflow UI — you'll use **Prompts**, **Experiments**, and **AI Gateway** tabs.

In [1]:
import difflib
import os
from pathlib import Path
from typing import Any, Literal

import mlflow
import requests
from loguru import logger
from mlflow.deployments import get_deploy_client
from mlflow.genai.optimize import MetaPromptOptimizer
from mlflow.genai.scorers import Guidelines, scorer
from openai import OpenAI
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


def find_docker_env(start: Path | None = None) -> Path:
    """Locate Docker.env from the notebook working directory or parents."""
    for directory in [start or Path.cwd(), *Path.cwd().parents]:
        candidate = directory / "Docker.env"
        if candidate.is_file():
            return candidate
    return Path.cwd() / "Docker.env"


def server_supports_workspaces(tracking_uri: str) -> bool:
    try:
        response = requests.get(
            f"{tracking_uri.rstrip('/')}/api/3.0/mlflow/server-info",
            timeout=5,
        )
        response.raise_for_status()
        return response.json().get("workspaces_enabled", False)
    except Exception as exc:
        logger.warning(f"Could not detect workspace support: {exc}")
        return False


def configure_mlflow_workspace(tracking_uri: str, workspace: str | None) -> str | None:
    """Activate a workspace when the server supports it."""
    if not workspace:
        os.environ.pop("MLFLOW_WORKSPACE", None)
        mlflow.set_workspace(None)
        return None
    if not server_supports_workspaces(tracking_uri):
        logger.info(
            f"Server has workspaces disabled; ignoring MLFLOW_WORKSPACE={workspace!r} for now"
        )
        os.environ.pop("MLFLOW_WORKSPACE", None)
        mlflow.set_workspace(None)
        return None
    os.environ["MLFLOW_WORKSPACE"] = workspace
    mlflow.set_workspace(workspace)
    return workspace


ENV_FILE = find_docker_env()
if not ENV_FILE.is_file():
    raise FileNotFoundError(
        f"Missing {ENV_FILE}. Copy Docker.env to the project root and edit your settings."
    )


In [2]:
import mlflow
import nest_asyncio
from dotenv import load_dotenv
from loguru import logger

load_dotenv(ENV_FILE, override=False)
nest_asyncio.apply()  # required for pydantic-ai in Jupyter

# --- Workshop configuration (from Docker.env) ---
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
GATEWAY_ENDPOINT = os.getenv("MLFLOW_GATEWAY_ENDPOINT", "stcruz-endpoint")
MLFLOW_WORKSPACE = os.getenv("MLFLOW_WORKSPACE", "default")
PROMPT_NAME = os.getenv("MLFLOW_PROMPT_NAME", "workshop-ticket-triage")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "mlflow-3-genai-workshop")
RUN_OPTIMIZATION = os.getenv("RUN_OPTIMIZATION", "true").lower() in {"1", "true", "yes"}

mlflow.set_tracking_uri(TRACKING_URI)
ACTIVE_WORKSPACE = configure_mlflow_workspace(TRACKING_URI, MLFLOW_WORKSPACE)
mlflow.set_experiment(EXPERIMENT_NAME)

gateway_client = get_deploy_client(TRACKING_URI)
openai_client = OpenAI(
    base_url=f"{TRACKING_URI.rstrip('/')}/gateway/mlflow/v1",
    api_key="unused",  # provider auth is handled by the gateway
)

logger.info(f"Env file: {ENV_FILE}")
logger.info(f"Tracking URI: {TRACKING_URI}")
logger.info(f"Workspace: {ACTIVE_WORKSPACE or 'legacy (workspaces disabled)'}")
logger.info(f"Gateway endpoint: {GATEWAY_ENDPOINT}")
logger.info(f"Experiment: {EXPERIMENT_NAME}")

2026-07-29 15:35:22.296 | INFO     | __main__:configure_mlflow_workspace:48 - Server has workspaces disabled; ignoring MLFLOW_WORKSPACE='default' for now
2026-07-29 15:35:22.458 | INFO     | __main__:<module>:27 - Env file: /Users/setiabudi/Documents/PROJECTS/LEARNING/mlflow_3_guidelines/prompt_registry_and_ai_gateway_using_mlflow3/Docker.env
2026-07-29 15:35:22.459 | INFO     | __main__:<module>:28 - Tracking URI: http://localhost:5000
2026-07-29 15:35:22.459 | INFO     | __main__:<module>:29 - Workspace: legacy (workspaces disabled)
2026-07-29 15:35:22.459 | INFO     | __main__:<module>:30 - Gateway endpoint: stcruz-endpoint
2026-07-29 15:35:22.459 | INFO     | __main__:<module>:31 - Experiment: mlflow-3-genai-workshop


---
## Part 1 — Prompt Registry

The Prompt Registry treats prompts as **versioned artifacts** (like models in a model registry).

**Concepts:**
- `register_prompt()` — create a prompt or a new version (same name → auto-increment)
- `load_prompt("prompts:/name@alias")` — load by version or alias
- `.format(...)` — fill `{{ variables }}`
- `set_prompt_alias()` — promote/rollback without code changes

### 1.1 Register version 1 — a minimal triage prompt

In [ ]:
V1_TEMPLATE = """Classify this support ticket into exactly one label:
PAYMENT, REFUND, RESCHEDULE, or OTHER.

Ticket: {{ ticket }}

Respond with the label only."""

prompt_v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=V1_TEMPLATE,
    commit_message="Workshop v1: minimal classification prompt",
    tags={"workshop": "mlflow-3.13", "task": "ticket-triage"},
)

print(f"Registered {prompt_v1.name} v{prompt_v1.version}")
print(f"URI: {prompt_v1.uri}")

2026/07/29 15:37:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: workshop-dummy-prompt, version 1


Registered workshop-dummy-prompt v1
URI: prompts:/workshop-dummy-prompt/1


### 1.2 Load the prompt and fill variables — no hardcoded prompt strings in app code

In [ ]:
loaded = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{prompt_v1.version}")

sample_ticket = "I was charged twice for my subscription this month."
filled = loaded.format(ticket=sample_ticket)

print("Formatted prompt preview:")
print(filled[:300], "...")

Formatted prompt preview:
Classify this support ticket into exactly one label:
PAYMENT, REFUND, RESCHEDULE, or OTHER.

Ticket: I was charged twice for my subscription this month.

Respond with the label only. ...


### 1.3 Register version 2 — add clearer rules (simulates a prompt improvement)

In [5]:
V2_TEMPLATE = """You are an expert support ticket triage assistant.

Task: Classify the ticket into exactly one of:
- PAYMENT — billing, duplicate charges, invoices
- REFUND — money back, cancellations, chargebacks
- RESCHEDULE — move/change appointment or delivery date
- OTHER — anything else

Rules:
- Output exactly one uppercase label
- No explanation, punctuation, or extra words

Ticket: {{ ticket }}"""

prompt_v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=V2_TEMPLATE,
    commit_message="Workshop v2: explicit label definitions and output rules",
)

print(f"Now at version {prompt_v2.version}")

2026/07/29 15:39:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: workshop-dummy-prompt, version 2


Now at version 2


### 1.4 Compare versions (like a Git diff)

In [6]:
v1 = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/1")
v2 = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/2")

print(f"v1: {v1.commit_message}")
print(f"v2: {v2.commit_message}\n")

diff = difflib.unified_diff(
    v1.template.splitlines(),
    v2.template.splitlines(),
    fromfile="v1",
    tofile="v2",
    lineterm="",
)
print("\n".join(diff))

v1: Workshop v1: minimal classification prompt
v2: Workshop v2: explicit label definitions and output rules

--- v1
+++ v2
@@ -1,6 +1,13 @@
-Classify this support ticket into exactly one label:
-PAYMENT, REFUND, RESCHEDULE, or OTHER.
+You are an expert support ticket triage assistant.
+
+Task: Classify the ticket into exactly one of:
+- PAYMENT — billing, duplicate charges, invoices
+- REFUND — money back, cancellations, chargebacks
+- RESCHEDULE — move/change appointment or delivery date
+- OTHER — anything else
+
+Rules:
+- Output exactly one uppercase label
+- No explanation, punctuation, or extra words
 
 Ticket: {{ ticket }}
-
-Respond with the label only.


### 1.5 Promote v2 to production via alias — apps load @production, not a version number

In [7]:
mlflow.genai.set_prompt_alias(
    PROMPT_NAME, alias="production", version=prompt_v2.version
)

production_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
print(f"Production alias → version {production_prompt.version}")

Production alias → version 2


**Check your work:** Open the MLflow UI → **Prompts** → `workshop-ticket-triage`. Compare v1 vs v2 and inspect the `production` alias.

---
## Part 2 — AI Gateway

The AI Gateway is a single **OpenAI-compatible** entry point. Your app references an **endpoint name**; the gateway holds provider credentials and routes requests.

Two ways to call it:
- **OpenAI SDK** — point `base_url` at the gateway
- **MLflow Deployments client** — `get_deploy_client(...).predict(...)

### 2.1 Discover configured endpoints (may be empty if using UI-only setup)

In [8]:
try:
    endpoints = gateway_client.list_endpoints()
    print("Available endpoints:")  
    for ep in endpoints:
        name = ep["name"] if isinstance(ep, dict) else ep
        print(f"  - {name}")
except Exception as exc:
    print(f"Could not list endpoints ({exc.__class__.__name__}).")
    print(f"Using configured endpoint: {GATEWAY_ENDPOINT}")

Could not list endpoints (HTTPError).
Using configured endpoint: stcruz-endpoint


### 2.2 Call the gateway with the OpenAI SDK — model = endpoint name

In [9]:
response = openai_client.chat.completions.create(
    model=GATEWAY_ENDPOINT,
    messages=[
        {"role": "user", "content": "In one sentence, what is the MLflow AI Gateway?"}
    ],
)
print(response.choices[0].message.content)

MLflow AI Gateway is a centralized service for securely managing, routing, and monitoring access to multiple AI model providers and endpoints through a single unified API.


### 2.3 Alternative: MLflow Deployments client (when /endpoints API is enabled)


In [10]:
question = "In one sentence, what is the MLflow Prompt Registry?"
messages = [{"role": "user", "content": question}]

try:
    result = gateway_client.predict(
        endpoint=GATEWAY_ENDPOINT,
        inputs={"messages": messages},
    )
    print(result["choices"][0]["message"]["content"])
except Exception as exc:
    print(f"Deployments API unavailable ({exc.__class__.__name__}).")
    print("Falling back to OpenAI-compatible gateway path (same endpoint):")
    fallback = openai_client.chat.completions.create(
        model=GATEWAY_ENDPOINT,
        messages=messages,
    )
    print(fallback.choices[0].message.content)

Deployments API unavailable (HTTPError).
Falling back to OpenAI-compatible gateway path (same endpoint):
MLflow Prompt Registry is a centralized place in MLflow to version, manage, and share prompts for LLM applications.


In [ ]:
# 2.4 Combine Prompt Registry + AI Gateway
def classify_ticket(ticket: str, prompt_uri: str | None = None) -> str:
    uri = prompt_uri or f"prompts:/{PROMPT_NAME}@production"
    prompt = mlflow.genai.load_prompt(uri)
    response = openai_client.chat.completions.create(
        model=GATEWAY_ENDPOINT,
        messages=[{"role": "user", "content": prompt.format(ticket=ticket)}],
    )
    return response.choices[0].message.content.strip()


for ticket in [
    "Charged twice for one hotel night.",
    "Need to move my appointment to next week.",
    "I want my money back for a cancelled flight.",
]:
    print(f"{ticket!r:55} → {classify_ticket(ticket)}")

**Check your work:** In the UI, open **AI Gateway** and confirm your endpoint. If usage tracking is enabled, you'll see traces after these calls.

---
## Part 3 — Pydantic AI + Gateway + Prompt Registry

[Pydantic AI](https://ai.pydantic.dev/) builds typed agents. Because the MLflow gateway is OpenAI-compatible, point Pydantic AI at the same `base_url` and use the endpoint name as the model id.

### 3.1 Create a gateway-backed Pydantic AI model


In [ ]:
pydantic_model = OpenAIChatModel(
    GATEWAY_ENDPOINT,
    provider=OpenAIProvider(
        base_url=f"{TRACKING_URI.rstrip('/')}/gateway/mlflow/v1",
        api_key="unused",
    ),
)

simple_agent = Agent(pydantic_model, system_prompt="You are a concise assistant.")
print(simple_agent.run_sync("Explain MLflow in one sentence.").output)

MLflow is an open-source platform for tracking, packaging, deploying, and managing machine learning experiments and models.


### 3.2 Structured output — Literal types work best with gateway-backed models


In [12]:
class TriageResult(BaseModel):
    category: Literal["PAYMENT", "REFUND", "RESCHEDULE", "OTHER"]
    reason: str


def triage_with_agent(
    ticket: str, prompt_uri: str = f"prompts:/{PROMPT_NAME}@production"
) -> TriageResult:
    """Load governed prompt from registry, call gateway-backed agent, return typed result."""
    prompt = mlflow.genai.load_prompt(prompt_uri)
    agent = Agent(
        pydantic_model,
        output_type=TriageResult,
        system_prompt="Classify support tickets. Follow the user message instructions exactly.",
    )
    return agent.run_sync(prompt.format(ticket=ticket)).output


ticket = "Charged twice for one hotel night."
triage_result = triage_with_agent(ticket)
print(f"Ticket: {ticket}")
print(f"Category: {triage_result.category}")
print(f"Reason:   {triage_result.reason}")

Ticket: Charged twice for one hotel night.
Category: PAYMENT
Reason:   Duplicate charge for a hotel night indicates a billing/payment issue.


**Design note:** The registry owns the *prompt template*; the gateway owns *model access*; Pydantic AI adds *typed outputs* and agent tooling. Production apps typically load `prompts:/.../@production` so prompt updates don't require redeploys.

---
## Part 4 — Prompt Evaluation

`mlflow.genai.evaluate()` runs your `predict_fn` over a dataset and scores outputs with custom or built-in scorers.

We'll compare behavior using:
- A **deterministic scorer** (exact label match)
- A **Guidelines scorer** (LLM-as-judge via the gateway)

In [13]:
EVAL_DATA = [
    {
        "inputs": {"ticket": "Charged twice for one hotel night."},
        "expectations": {"expected_response": "PAYMENT"},
    },
    {
        "inputs": {"ticket": "Need to move my appointment to next week."},
        "expectations": {"expected_response": "RESCHEDULE"},
    },
    {
        "inputs": {"ticket": "I want my money back for a cancelled flight."},
        "expectations": {"expected_response": "REFUND"},
    },
    {
        "inputs": {"ticket": "What are your opening hours on weekends?"},
        "expectations": {"expected_response": "OTHER"},
    },
]

print(f"Evaluation dataset: {len(EVAL_DATA)} examples")

Evaluation dataset: 4 examples


In [ ]:
@scorer
def category_match(outputs: Any, expectations: dict[str, Any]) -> float:
    predicted = str(outputs).strip().upper().strip(".")
    expected = expectations["expected_response"]
    return float(predicted == expected)


def make_predict_fn(prompt_uri: str):
    def predict_fn(ticket: str) -> str:
        prompt = mlflow.genai.load_prompt(prompt_uri)
        return (
            openai_client.chat.completions.create(
                model=GATEWAY_ENDPOINT,
                messages=[{"role": "user", "content": prompt.format(ticket=ticket)}],
            )
            .choices[0]
            .message.content.strip()
        )

    return predict_fn


label_guidelines = Guidelines(
    name="label_only",
    guidelines="The response must be exactly one uppercase label (PAYMENT, REFUND, RESCHEDULE, or OTHER) with no extra words.",
    model=f"gateway:/{GATEWAY_ENDPOINT}",
)

In [ ]:
# Evaluate v1 vs v2 side by side
results = {}
for version in [1, 2]:
    uri = f"prompts:/{PROMPT_NAME}/{version}"
    with mlflow.start_run(run_name=f"eval-v{version}") as run:
        eval_result = mlflow.genai.evaluate(
            data=EVAL_DATA,
            predict_fn=make_predict_fn(uri),
            scorers=[category_match, label_guidelines],
        )
        results[version] = eval_result
        print(f"v{version} metrics: {eval_result.metrics}")
        print(
            f"  → {TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}\n"
        )

**Check your work:** Open the evaluation runs in the MLflow UI. Compare `category_match/mean` and the per-row results table.

---
## Part 5 — Prompt Optimization

`mlflow.genai.optimize_prompts()` automatically rewrites a registered prompt to improve scorer metrics, saving the result as a **new registry version**.

For workshops we use **MetaPromptOptimizer** (fast, works with a handful of examples). For production with 100+ labeled rows, consider `GepaPromptOptimizer`.

In [ ]:
if RUN_OPTIMIZATION:
    baseline = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/1")

    optimization_result = mlflow.genai.optimize_prompts(
        predict_fn=make_predict_fn(baseline.uri),
        train_data=EVAL_DATA,
        prompt_uris=[baseline.uri],
        optimizer=MetaPromptOptimizer(reflection_model=f"gateway:/{GATEWAY_ENDPOINT}"),
        scorers=[category_match],
    )

    optimized = optimization_result.optimized_prompts[0]
    print(f"Baseline score : {optimization_result.initial_eval_score}")
    print(f"Optimized score: {optimization_result.final_eval_score}")
    print(f"New prompt URI : {optimized.uri} (v{optimized.version})")
    print("\nOptimized template preview:")
    print(optimized.template[:500], "...")
else:
    print("Skipped optimization (RUN_OPTIMIZATION=False)")

In [ ]:
if RUN_OPTIMIZATION:
    # Promote the optimized version — one line, no redeploy
    mlflow.genai.set_prompt_alias(
        PROMPT_NAME, alias="production", version=optimized.version
    )

    # Re-evaluate production alias on the same dataset
    with mlflow.start_run(run_name="eval-production-after-optimize") as run:
        final_eval = mlflow.genai.evaluate(
            data=EVAL_DATA,
            predict_fn=make_predict_fn(f"prompts:/{PROMPT_NAME}@production"),
            scorers=[category_match, label_guidelines],
        )
        print(f"Production metrics: {final_eval.metrics}")
        print(
            f"View run: {TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
        )

---
## Workshop Recap

You just walked through the full GenAI governance loop:

| Step | API / Tool | What it governs |
| --- | --- | --- |
| Register & version | `mlflow.genai.register_prompt()` | Prompt wording |
| Promote / rollback | `mlflow.genai.set_prompt_alias()` | Which version is live |
| Model access | AI Gateway endpoint | Provider, credentials, routing |
| Typed agents | Pydantic AI + gateway `base_url` | Application structure |
| Quality measurement | `mlflow.genai.evaluate()` | Objective prompt comparison |
| Auto-improvement | `mlflow.genai.optimize_prompts()` | Data-driven prompt rewrites |

**Suggested exercises:**
1. Register a v3 prompt manually and A/B evaluate it against `@production`
2. Add a custom `@scorer` for brevity (penalize long responses)
3. Enable usage tracking on your gateway endpoint and inspect traces
4. Swap `MetaPromptOptimizer` for `GepaPromptOptimizer` with a larger dataset

**References:**
- [Prompt Registry docs](https://mlflow.org/docs/latest/genai/prompt-registry/)
- [AI Gateway docs](https://mlflow.org/docs/latest/genai/governance/ai-gateway/)
- [Optimize prompts](https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/)